# Week 22 · Notebook 2: Streaming & Lakeflow Pipelines (formerly Delta Live Tables)

# Requirements: Databricks workspace (free trial) + upload the week-01 CSVs to a volume

Upload into `/Volumes/zrl_/zorologistics/raw/`:

- `shipments.csv`

**Also** drop a copy of `shipments.csv` into `/Volumes/zrl_/zorologistics/raw/events/`, that folder is the Auto Loader source. To watch the stream grow, drop more CSV batches into `events/` and re-run the pipeline.

Run this notebook as a **Lakeflow Pipeline** (Jobs & Pipelines → Pipeline). Set the target schema to `zrl_.zorologistics` (or a dedicated schema if names collide with Week 21 tables).


## Lakeflow Pipelines concepts

Lakeflow Pipelines (formerly **Delta Live Tables**) is the declarative layer: you *declare* datasets, and the pipeline resolves the DAG, ordering, and retries. Three dataset kinds:

- **Streaming table**: each record processed once; incremental; append-only source. Use for ingestion.
- **Materialized view**: recomputed to reflect current state ("always correct"). Use for aggregations/joins.
- **View**: evaluated on demand, not persisted. Use for intermediate checks.

**Expectations** are data-quality gates: `@dp.expect` (warn, keep row), `@dp.expect_or_drop` (remove bad row), `@dp.expect_or_fail` (stop the update). See `reference/platforms/databricks/06-pipelines-jobs.md` and research §5.3.


In [ ]:
# Lakeflow Pipelines, CURRENT Python API.
# Legacy code says: import dlt
# Current code says: from pyspark import pipelines as dp
from pyspark import pipelines as dp
from pyspark.sql import functions as F

print("Lakeflow Pipelines module:", dp)


## Bronze: streaming table via Auto Loader

`spark.readStream.format("cloudFiles")` is **Auto Loader**: it incrementally ingests new files with exactly-once semantics. In a pipeline the checkpoint is managed for you; `cloudFiles.schemaLocation` only pins schema inference. Dataset functions must return a DataFrame and must **not** call actions (`collect`/`count`/`toPandas`/`save`).


In [ ]:
@dp.table
def bronze_events():
    """Streaming table: incrementally ingests shipment CSVs dropped into the events folder."""
    return (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .option("cloudFiles.schemaLocation", "/Volumes/zrl_/zorologistics/checkpoints/bronze_events")
        .load("/Volumes/zrl_/zorologistics/raw/events/")
    )


## Silver: materialized view with expectations

`dp.read("bronze_events")` reads the streaming table's current snapshot. The two decorators stack on the same function: `@dp.expect_or_drop` removes rows violating the weight constraint; `@dp.expect` warns (keeps the row, records the metric) on an unexpected status.


In [ ]:
@dp.materialized_view
@dp.expect_or_drop("weight_positive", "weight_kg > 0")
@dp.expect("status_valid", "status IN ('Delivered', 'In Transit', 'Booked')")
def silver():
    return (
        dp.read("bronze_events")
        .dropDuplicates(["shipment_id"])
        .withColumn("weight_kg", F.col("weight_kg").cast("double"))
        .withColumn("delay_hours", F.col("delay_hours").cast("double"))
        .withColumn("planned_departure",
                    F.to_timestamp("planned_departure", "yyyy-MM-dd HH:mm:ss[.SSSSSS]"))
        .withColumn("actual_arrival",
                    F.to_timestamp("actual_arrival", "yyyy-MM-dd HH:mm:ss[.SSSSSS]"))
        .withColumn("is_on_time", F.col("delay_hours") <= 2.0)
    )


## Gold: materialized view aggregating silver

The same on-time KPI aggregation as Week 21, now recomputed automatically whenever silver changes.


In [ ]:
@dp.materialized_view
def gold_on_time_kpis():
    return (
        dp.read("silver")
        .withColumn("month", F.date_format("actual_arrival", "yyyy-MM"))
        .groupBy("carrier_id", "lane_id", "month")
        .agg(
            F.count("*").alias("shipment_count"),
            F.round(F.avg(F.when(F.col("is_on_time"), 1.0).otherwise(0.0)), 4).alias("on_time_rate"),
            F.round(F.avg("delay_hours"), 2).alias("avg_delay_hours"),
        )
    )


In [ ]:
# A temporary view is an intermediate check that is not persisted.
@dp.temporary_view
def status_distribution():
    return dp.read("silver").groupBy("status").count()


## The same pipeline in SQL (comparison)

Lakeflow Pipelines also accept SQL: `STREAM` + `read_files` for a streaming table, and a `MATERIALIZED VIEW` with a `CONSTRAINT … EXPECT … ON VIOLATION …` quality clause. The constraint below maps to the Python `@dp.expect_or_drop`.


In [ ]:
%sql
-- SQL form of the same pipeline (STREAM + read_files, materialized view + constraint).
CREATE OR REFRESH STREAMING TABLE bronze_events_sql AS
SELECT * FROM STREAM read_files('/Volumes/zrl_/zorologistics/raw/events/',
                                format => 'csv', header => true);

CREATE OR REFRESH MATERIALIZED VIEW silver_sql
CONSTRAINT weight_positive EXPECT (weight_kg > 0) ON VIOLATION DROP ROW
AS
SELECT * FROM bronze_events_sql


## Run it & read the quality metrics

1. In the UI: **Jobs & Pipelines → Pipelines → Create pipeline**, choose this notebook, set the target schema, and run.
2. The **event log** shows each dataset's updates and, for expectations, how many rows were *kept / dropped / failed*.
3. `bronze_events` and `silver` appear as governed Delta tables with their own lineage and time travel.


In [ ]:
# Final metric (run interactively AFTER the pipeline completes, or in dev mode).
try:
    n = spark.sql("SELECT count(*) FROM zrl_.zorologistics.silver").collect()[0][0]
    print("silver rows:", n)
except Exception as e:
    # Before the pipeline runs, silver does not exist yet, print a deterministic placeholder.
    print("silver rows: 0 (pipeline not yet run; the metric populates after the first update)")
    print("reason:", str(e)[:120])
